# DynamoDB Data Extraction

In [ ]:

import boto3
import json
import os
from dotenv import load_dotenv

REGION   = 'ap-southeast-2'
MODEL_ID = 'anthropic.claude-3-5-sonnet-20241022-v2:0'

In [ ]:
load_dotenv(dotenv_path=r'/home/sagemaker-user/swtest1/llm_poc/.env', override=True)
DEV_ROLE = os.getenv('DEV_ROLE')

# Remove any AWS credentials the .env may have injected
for key in ['AWS_BEARER_TOKEN_BEDROCK', 'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_SESSION_TOKEN']:
    os.environ.pop(key, None)

print(f'DEV_ROLE loaded: {bool(DEV_ROLE)}')


In [ ]:

def get_dynamodb_client():
    """Fresh STS assume-role + DynamoDB client. Call each time to avoid token expiry."""
    sts = boto3.client('sts')
    session = sts.assume_role(RoleArn=DEV_ROLE, RoleSessionName='notebook-session')
    creds = session['Credentials']
    return boto3.client(
        'dynamodb',
        region_name=REGION,
        aws_access_key_id=creds['AccessKeyId'],
        aws_secret_access_key=creds['SecretAccessKey'],
        aws_session_token=creds['SessionToken'],
    )

def lookup_merchant(merchant_name: str) -> dict:
    normalised = merchant_name.upper().strip()
    escaped    = normalised.translate(str.maketrans({"'": "''"}))
    try:
        client = get_dynamodb_client()
        resp   = client.execute_statement(
            Statement=(
                "SELECT * FROM mapping "
                "WHERE \"type\" = 'merchant' "
                f"AND \"entity\" = '{escaped}'"
            )
        )
        items = resp.get('Items', [])
        if items:
            def unwrap(v):
                return next(iter(v.values())) if isinstance(v, dict) and v else None
            item = items[0]
            return {
                'found':    True,
                'entity':   unwrap(item.get('entity')),
                'category': unwrap(item.get('category')),
                'mapping':  unwrap(item.get('mapping')),
            }
        return {'found': False, 'entity': normalised}
    except Exception as e:
        return {'found': False, 'error': str(e)}

## Tests

In [ ]:
print('\n--- DynamoDB ---')
for merchant in ['WOOLWORTHS', 'NETFLIX', 'CENTRELINK']:
    result = lookup_merchant(merchant)
    print(f'{merchant}: {result}')

In [ ]:
def get_distinct_categories(client) -> list:
    """Scan mapping table and return distinct category values."""
    categories = set()
    kwargs = {'Statement': 'SELECT category FROM mapping WHERE "type" = \'merchant\''}
    
    while True:
        resp = client.execute_statement(**kwargs)
        for item in resp.get('Items', []):
            cat = item.get('category', {})
            val = next(iter(cat.values()), None) if cat else None
            if val:
                categories.add(val)
        next_token = resp.get('NextToken')
        if not next_token:
            break
        kwargs['NextToken'] = next_token
    
    return sorted(categories)



In [ ]:
categories = get_distinct_categories(get_dynamodb_client())
for c in categories:
    print(c)

In [ ]:
from collections import Counter

def count_by_category(target_categories: list, client) -> dict:
    """Fast count of merchant entries per category."""
    counts = Counter()
    kwargs = {'Statement': "SELECT category FROM mapping WHERE \"type\" = 'merchant'"}
    
    while True:
        resp = client.execute_statement(**kwargs)
        for item in resp.get('Items', []):
            cat = next(iter(item.get('category', {}).values()), None)
            if cat in target_categories:
                counts[cat] += 1
        next_token = resp.get('NextToken')
        if not next_token:
            break
        kwargs['NextToken'] = next_token
    
    return dict(counts)



In [ ]:
targets = ['HEALTHCARE_MEDICAL', 'INSURANCE_MEDICAL', 'GYMS___FITNESS', 'PETS_PET_CARE']
counts = count_by_category(targets, get_dynamodb_client())
for cat, n in sorted(counts.items()):
    print(f'{cat}: {n:,}')

In [ ]:
kwargs = {'Statement': "SELECT * FROM mapping WHERE \"type\" = 'merchant'"}

In [ ]:
import pandas as pd

In [ ]:
def fetch_category_merchants(target_categories: list, client) -> pd.DataFrame:
    """Fetch all fields for merchant entries matching target categories."""
    rows = []
    kwargs = {'Statement': "SELECT * FROM mapping WHERE \"type\" = 'merchant'"}
    
    while True:
        resp = client.execute_statement(**kwargs)
        for item in resp.get('Items', []):
            cat = next(iter(item.get('category', {}).values()), None)
            if cat in target_categories:
                rows.append({
                    k: next(iter(v.values()), None)
                    for k, v in item.items()
                })
        next_token = resp.get('NextToken')
        if not next_token:
            break
        kwargs['NextToken'] = next_token
    
    df = pd.DataFrame(rows)
    df['tier'] = df['category'].map({
        'HEALTHCARE_MEDICAL': 1, 'INSURANCE_MEDICAL': 1,
        'GYMS___FITNESS': 2,     'PETS_PET_CARE': 2,
    })
    return df.sort_values(['tier', 'category', 'entity']).reset_index(drop=True)



In [ ]:
targets = ['HEALTHCARE_MEDICAL', 'INSURANCE_MEDICAL', 'GYMS___FITNESS', 'PETS_PET_CARE']
df = fetch_category_merchants(targets, get_dynamodb_client())
print(df.shape)
print(df.columns.tolist())
df.head(20)

In [ ]:
df.to_csv('health_combined.csv', index=False)

In [ ]:
df[df.category=='HEALTHCARE_MEDICAL'].to_csv('health_medical.csv', index=False)

In [ ]:
def apply_multi_category(df):
    # 1. Initialize the local dictionary with the required keys
    local_dictionary = {
        'amount': df["amount"],
        'category': DEFAULT_CATEGORY
    }

    if pd.isnull(df[COL_RULE_ENRICH]):
        local_dictionary['category'] = df[COL_BASE_CAT_ENRICH]
    elif df[COL_RULE_ENRICH].lower() in ("none", None):
        local_dictionary['category'] = df[COL_BASE_CAT_ENRICH]
    else:
        try:
            # 2. Pass the empty globals {} and the local_dictionary into exec()
            exec(df[COL_RULE_ENRICH].replace("\\n", "\n"), {}, local_dictionary)

            df[COL_ENRICH_STEPS] += STEP_SPEND_APPLY_RULES
        except Exception as e:
            raise InvalidRuleError(
                f"Invalid rule: {df[COL_RULE_ENRICH]} with exception: {e}"
            )

    # 3. Extract the final category safely from the dictionary
    final_category = local_dictionary['category']

    if isinstance(final_category, str):
        return final_category.replace('"', "").replace('"', "")
    else:
        return final_category